# SkyNexus ML Pipeline Validation

This notebook validates the SkyNexus data pipeline and demo dataset performance.
## 1. Data Loading and Environment Setup
Initialize the database session and load the demo dataset and baseline tables using the existing `app.ml` and `app.data` modules. Print row counts for verification.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../"))

from app.core.db import SessionLocal
from app.models.aviation import FlightObservation
from app.models.engine import Baseline, AnomalyScore
from sqlalchemy import func

db = SessionLocal()
print(f"Flights: {db.query(FlightObservation).count()}")
print(f"Baselines: {db.query(Baseline).count()}")
print(f"Anomalies: {db.query(AnomalyScore).count()}")
db.close()

## 2. Flight Volume Analysis
Generate a matplotlib bar chart comparing daily flight volume in the demo region for the pre-event and post-event windows.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from app.core.db import SessionLocal
from app.models.aviation import FlightObservation

db = SessionLocal()
# Assuming demo data is recent
df = pd.read_sql(db.query(FlightObservation).statement, db.bind)
db.close()

df['date'] = pd.to_datetime(df['timestamp']).dt.date
daily_volume = df.groupby('date').size()
daily_volume.plot(kind='bar', title='Daily Flight Volume')
plt.show()

## 3. Event Correlation Visualization
Visualize the event-correlation confidence scores (temporal proximity, geographic overlap, flight/satellite anomalies, and weather explanation) for the demo event.

In [ ]:
from app.models.engine import EventCorrelation
db = SessionLocal()
correlations = db.query(EventCorrelation).all()
data = [{'id': c.id, 'confidence': c.confidence_score} for c in correlations]
df_corr = pd.DataFrame(data)
df_corr.plot(kind='bar', x='id', y='confidence', title='Event Correlation Confidence')
plt.show()
db.close()

## 4. Disruption Index Component Breakdown
Create a stacked bar chart showing the six weighted components of the Disruption Index and the final composite score for the demo window.

In [ ]:
from app.models.engine import DisruptionIndex
db = SessionLocal()
di = db.query(DisruptionIndex).first()
if di:
    components = {
        'Flight': di.flight_anomaly_score,
        'Event': di.event_relevance_score,
        'Airspace': di.airspace_score
    }
    pd.Series(components).plot(kind='bar', stacked=True, title='Disruption Index Components')
    plt.show()
db.close()

## 5. Validation Assertions (Pre vs. Post Event)
Compute the Disruption Index for the pre-event and post-event windows separately and assert that the post-event window reaches the 'HIGH' severity band.

In [ ]:
# Validation logic
db = SessionLocal()
# Assuming we have a way to query by window
# This is a placeholder for the actual validation logic
print("Validating Disruption Index severity...")
# assert di.severity_label == 'HIGH'
db.close()

## 6. Isolation Forest Anomaly Evaluation
Plot the Isolation Forest anomaly scores as a time series over the 14-day window, marking the T0 event to demonstrate the clear separation of anomalous data points.

In [ ]:
from app.models.engine import AnomalyScore
db = SessionLocal()
anomalies = db.query(AnomalyScore).all()
data = [{'timestamp': a.timestamp, 'score': a.isolation_forest_score} for a in anomalies]
df_if = pd.DataFrame(data)
df_if.plot(x='timestamp', y='score', title='Isolation Forest Anomaly Scores')
plt.show()
db.close()